# 05 Ablation Sweeps

Run a large grid of SDD variants and record results.

In [ ]:

from pathlib import Path
import os, json, yaml, math, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader, random_split, Subset

from src.utils.seed import set_seed
from src.models.unet import UNetModel
from src.training.trainer import SDDTrainer
from src.evaluation.feature_extract import extract_features
from src.evaluation.linear_probe import train_linear_probe
from src.evaluation.fid import FIDEvaluator

ROOT = Path('.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(42)


In [ ]:

import yaml, copy, itertools, time
from pathlib import Path

def load_cfg(path):
    return yaml.safe_load(Path(path).read_text())

def make_loaders(cfg):
    size = cfg["dataset"]["image_size"]
    train_tfm = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    eval_tfm = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    if cfg["dataset"]["name"] == "cifar10":
        train_ds = datasets.CIFAR10(root=cfg["dataset"]["root"], train=True, download=True, transform=train_tfm)
        test_ds = datasets.CIFAR10(root=cfg["dataset"]["root"], train=False, download=True, transform=eval_tfm)
    else:
        train_ds = datasets.ImageFolder(root=cfg["dataset"]["train_dir"], transform=train_tfm)
        test_ds = datasets.ImageFolder(root=cfg["dataset"]["val_dir"], transform=eval_tfm)
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["train"]["batch_size"],
        shuffle=True,
        num_workers=cfg["dataset"]["num_workers"],
        pin_memory=True,
        drop_last=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg["train"]["batch_size"],
        shuffle=False,
        num_workers=cfg["dataset"]["num_workers"],
        pin_memory=True,
        drop_last=False,
    )
    return train_loader, test_loader

def build_model(cfg):
    return UNetModel(
        base_channels=cfg["model"]["channels"],
        channel_mults=tuple(cfg["model"]["channel_mults"]),
        num_res_blocks=cfg["model"]["num_res_blocks"],
        attention_resolutions=tuple(cfg["model"]["attention_resolutions"]),
        dropout=cfg["model"]["dropout"],
        image_size=cfg["dataset"]["image_size"],
    )

def make_optimizer(trainer, cfg):
    params = list(trainer.model.parameters())
    if getattr(trainer, "proj_student", None) is not None:
        params += list(trainer.proj_student.parameters())
    return torch.optim.AdamW(params, lr=cfg["train"]["lr"], weight_decay=cfg["train"]["weight_decay"])

def maybe_init_wandb(cfg, name=None):
    if not cfg.get("wandb", {}).get("enabled", False):
        return None
    import wandb
    return wandb.init(
        project=cfg["wandb"]["project"],
        entity=cfg["wandb"].get("entity"),
        name=name or cfg.get("run_name"),
        config=cfg,
        tags=cfg["wandb"].get("tags", []),
        reinit=True,
    )

def save_cfg(cfg, path):
    Path(path).write_text(yaml.safe_dump(cfg, sort_keys=False))

def train_epochs(trainer, loader, cfg, optimizer, run=None, epochs=None, ckpt_dir="outputs/checkpoints"):
    ckpt_dir = Path(ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg["train"]["mixed_precision"] and torch.cuda.is_available())
    epochs = epochs or cfg["train"]["epochs"]
    history = []
    for epoch in range(epochs):
        trainer.state.epoch = epoch
        metrics = trainer.train_one_epoch(loader, optimizer, scaler=scaler, wandb_run=run)
        history.append({"epoch": epoch, **metrics})
        if (epoch + 1) % cfg["train"]["save_every"] == 0:
            torch.save({
                "model": trainer.model.state_dict(),
                "teacher": None if trainer.teacher is None else trainer.teacher.state_dict(),
                "proj_student": None if trainer.proj_student is None else trainer.proj_student.state_dict(),
                "proj_teacher": None if trainer.proj_teacher is None else trainer.proj_teacher.state_dict(),
                "epoch": epoch,
                "cfg": cfg,
            }, ckpt_dir / f"{cfg.get('run_name', 'run')}_epoch{epoch+1}.pt")
    return pd.DataFrame(history)


In [ ]:

ablation_grid = [
    ("full", {}),
    ("no_centering", {"use_centering": False}),
    ("no_sharpening", {"use_sharpening": False}),
    ("no_gating", {"use_gating": False}),
    ("no_projection_head", {"use_projection_head": False}),
    ("mse_only", {"enabled": False, "lambda_distill": 0.0}),
    ("ema_only", {"use_centering": False, "use_sharpening": False, "use_gating": False, "use_projection_head": False}),
    ("soft_gating", {"gating": {"mode": "soft", "t_min": 0.1, "t_max": 0.6, "soft_mid": 0.4, "soft_beta": 0.08}}),
]

results = []
for tag, overrides in ablation_grid:
    cfg = load_cfg("configs/cifar10.yaml")
    cfg["run_name"] = f"cifar10_{tag}"
    cfg["wandb"]["enabled"] = True
    cfg["wandb"]["tags"] = ["cifar10", "ablation", tag]
    for k, v in overrides.items():
        if k == "gating":
            cfg["sdd"]["gating"].update(v)
        elif k in cfg["sdd"]:
            cfg["sdd"][k] = v
        else:
            cfg["sdd"].update({k: v})
    train_loader, _ = make_loaders(cfg)
    model = build_model(cfg)
    trainer = SDDTrainer(model, cfg, DEVICE)
    optimizer = make_optimizer(trainer, cfg)
    run = maybe_init_wandb(cfg, name=cfg["run_name"])
    hist = train_epochs(trainer, train_loader, cfg, optimizer, run=run, epochs=5)
    results.append({
        "tag": tag,
        "loss_total": hist["loss_total"].iloc[-1],
        "loss_mse": hist["loss_mse"].iloc[-1],
        "loss_distill": hist["loss_distill"].iloc[-1],
        "gate_mean": hist["gate_mean"].iloc[-1],
    })

ablation_df = pd.DataFrame(results)
ablation_df.sort_values("loss_total")


In [ ]:

ablation_df.to_csv("outputs/ablation_summary.csv", index=False)
ablation_df
